#This notebook covers Multi-Country 2026 Revenue Forecast Dashboards: **Daily and Monthly Actual vs Forecast (Bahrain, UAE, Egypt, and Oman)**

#SECTION 1: Import Libraries & Connect to Google Drive & Import Datasets

In [49]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FOLDER = '/content/drive/MyDrive/Colab Notebooks/Thesis/Thesis_Datasets_Forecast'

files = {
    'Bahrain (AFS)'   : f'{FOLDER}/BH_AFS_forecast_2026.xlsx',
    'Bahrain (AUB)'   : f'{FOLDER}/BH_AUB_forecast_2026.xlsx',
    'UAE (AFS)'       : f'{FOLDER}/UAE_AFS_forecast_2026.xlsx',
    'Egypt (AFS)'     : f'{FOLDER}/EG_AFS_forecast_2026.xlsx',
    'Oman (SOH)'      : f'{FOLDER}/OM_SOH_forecast_2026.xlsx',
    'Oman (ABO)'      : f'{FOLDER}/OM_ABO_forecast_2026.xlsx',
}

# --- Currency per entity ---
currencies = {
    'Bahrain (AFS)' : 'BHD',
    'Bahrain (AUB)' : 'BHD',
    'UAE (AFS)'     : 'AED',
    'Egypt (AFS)'   : 'EGP',
    'Oman (SOH)'    : 'OMR',
    'Oman (ABO)'    : 'OMR',
}

datasets = {}
for name, path in files.items():
    df = pd.read_excel(path)
    df['Settlement Date'] = pd.to_datetime(df['Settlement Date'])
    df = df.sort_values('Settlement Date').reset_index(drop=True)
    df_2026 = df[df['Settlement Date'].dt.year == 2026].copy()
    datasets[name] = df_2026
    print(f'{name:<18} ({currencies[name]})  {len(df_2026)} rows | '
          f'{df_2026["Settlement Date"].min().date()} to {df_2026["Settlement Date"].max().date()} ')

positions = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bahrain (AFS)      (BHD)  365 rows | 2026-01-01 to 2026-12-31 
Bahrain (AUB)      (BHD)  365 rows | 2026-01-01 to 2026-12-31 
UAE (AFS)          (AED)  365 rows | 2026-01-01 to 2026-12-31 
Egypt (AFS)        (EGP)  365 rows | 2026-01-01 to 2026-12-31 
Oman (SOH)         (OMR)  365 rows | 2026-01-01 to 2026-12-31 
Oman (ABO)         (OMR)  365 rows | 2026-01-01 to 2026-12-31 


#SECTION 2: Daily Dashboards (6 subplots for All Entities)

In [50]:
fig_daily = make_subplots(
    rows=3, cols=2,
    subplot_titles=list(datasets.keys()),
    vertical_spacing=0.08, horizontal_spacing=0.10
)

for (name, df), (r, c) in zip(datasets.items(), positions):
    ccy = currencies[name]
    actual = df[df['Type'] == 'Actual']
    forecast = df[df['Type'] == 'Forecast']

    fig_daily.add_trace(go.Scatter(
        x=actual['Settlement Date'], y=actual['Profit_Best'],
        mode='lines', name='Actual', line=dict(color='#2B6CB0', width=1.3),
        legendgroup='actual', showlegend=(r==1 and c==1),
        hovertemplate=f'%{{x|%b %d, %Y}}<br>%{{y:,.0f}} {ccy}<extra></extra>'
    ), row=r, col=c)

    fig_daily.add_trace(go.Scatter(
        x=forecast['Settlement Date'], y=forecast['Profit_Best'],
        mode='lines', name='Forecast', line=dict(color='#DD6B20', width=1.3, dash='dash'),
        legendgroup='forecast', showlegend=(r==1 and c==1),
        hovertemplate=f'%{{x|%b %d, %Y}}<br>%{{y:,.0f}} {ccy}<extra></extra>'
    ), row=r, col=c)

    fig_daily.update_yaxes(title_text=f'{ccy}', row=r, col=c, tickformat=',.0f')

fig_daily.update_layout(
    title='2026 Daily Revenue — Actual vs Forecast (All Entities)',
    height=1300, width=1300,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(orientation='h', y=1.05, x=0.5, xanchor='center'),
    margin=dict(t=100)
)
fig_daily.show()

#Section 3: Monthly Dashboards (6 subplots for All Entities)

In [51]:
fig_monthly = make_subplots(
    rows=3, cols=2,
    subplot_titles=list(datasets.keys()),
    vertical_spacing=0.08, horizontal_spacing=0.10
)

for (name, df), (r, c) in zip(datasets.items(), positions):
    ccy = currencies[name]
    actual_m = df[df['Type']=='Actual'].set_index('Settlement Date')['Profit_Best'].resample('MS').sum()
    forecast_m = df[df['Type']=='Forecast'].set_index('Settlement Date')['Profit_Best'].resample('MS').sum()

    if len(actual_m) > 0:
        last_actual_date = df[df['Type']=='Actual']['Settlement Date'].max()
        if last_actual_date.day < 28:
            cutoff = last_actual_date.replace(day=1)
            # --- Capture the transition month's real actual total BEFORE dropping it ---
            transition_month_actual = actual_m[actual_m.index == cutoff]
            actual_m = actual_m[actual_m.index < cutoff]

            # --- Add those real days into forecast_m's same month, so it reflects
            #     a genuine full-month total (real days + forecasted days)
            if len(transition_month_actual) > 0 and cutoff in forecast_m.index:
                forecast_m.loc[cutoff] = forecast_m.loc[cutoff] + transition_month_actual.iloc[0]

    if len(actual_m) > 0 and len(forecast_m) > 0:
        last_point = pd.Series({actual_m.index[-1]: actual_m.iloc[-1]})
        forecast_m_connected = pd.concat([last_point, forecast_m]).sort_index()
    else:
        forecast_m_connected = forecast_m

    fig_monthly.add_trace(go.Scatter(
        x=actual_m.index, y=actual_m.values,
        mode='lines+markers', name='Actual', line=dict(color='#2B6CB0', width=2),
        marker=dict(size=5), legendgroup='actual', showlegend=(r==1 and c==1),
        hovertemplate=f'%{{x|%b %Y}}<br>%{{y:,.0f}} {ccy}<extra></extra>'
    ), row=r, col=c)

    fig_monthly.add_trace(go.Scatter(
        x=forecast_m_connected.index, y=forecast_m_connected.values,
        mode='lines+markers', name='Forecast', line=dict(color='#DD6B20', width=2, dash='dash'),
        marker=dict(size=5), legendgroup='forecast', showlegend=(r==1 and c==1),
        hovertemplate=f'%{{x|%b %Y}}<br>%{{y:,.0f}} {ccy}<extra></extra>'
    ), row=r, col=c)

    fig_monthly.update_yaxes(title_text=f'{ccy}', row=r, col=c, tickformat=',.0f')

fig_monthly.update_layout(
    title='2026 Monthly Revenue — Actual vs Forecast (All Entities)',
    height=1300, width=1300,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(orientation='h', y=1.05, x=0.5, xanchor='center'),
    margin=dict(t=100, l=80)
)
fig_monthly.show()